# Linly-Dubbing Colab WebUI
This notebook is optimized for running **Linly-Dubbing** in Google Colab with T4 GPU.

### 🛠️ Execution Guide
1.  **Step 1**: Initialize Conda environment (Kernel will restart).
2.  **Step 2**: Clone repository and submodules.
3.  **Step 3**: Install system and Python dependencies.
4.  **Step 4**: Apply runtime patches and download models.
5.  **Step 5**: Launch the WebUI.

In [ ]:
# [Step 1] 初始化 Conda 环境 (Initialize Conda)
# 注意：执行此单元格后内核会自动重启 (Notebook will restart after execution)
try:
    import condacolab
    condacolab.check()
    print("Conda environment already initialized.")
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

✨🍰✨ Everything looks OK!
Conda environment already initialized.


In [ ]:
# [Step 2] 获取代码 (Get Code)
import os
if not os.path.exists('/content/Linly-Dubbing'):
    %cd /content/
    !git clone https://github.com/infinite-gaming-studio/Linly-Dubbing.git --depth 1
else:
    print('Project already cloned. Pulling latest changes...')
    %cd /content/Linly-Dubbing
    !git pull

%cd /content/Linly-Dubbing
!git submodule update --init --recursive

Project already cloned. Pulling latest changes...
/content/Linly-Dubbing
Already up to date.
/content/Linly-Dubbing


In [ ]:
# [Step 1.4] Python 依赖安装 (Python Dependencies via uv)
# 我们使用 'uv' 进行极速安装。Colab 环境需要特定版本的库以保证 ASR/TTS 兼容性。

# 1. 安装 uv
!pip install -q uv

# 2. [CRITICAL] 强制重装核心库 (Fix for numpy, setuptools, torch compatibility)
print("Installing core compute libraries... This might take a minute.")
# numpy<2.0.0 and torch==2.3.1 are strictly required
!uv pip install --system --force-reinstall "numpy<2.0.0" "setuptools" "loguru" "yt-dlp" torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# 3. 运行时修补 (Runtime Patching)
# 将 numpy==1.26.3 修改为 numpy<2.0.0 以提高灵活性
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt

# [CRITICAL] Patch TTS submodule for Python 3.12 support (Colab uses 3.12)
import os
if os.path.exists('submodules/TTS/setup.py'):
    !sed -i 's/if Version(python_version) < Version("3.9") or Version(python_version) >= Version("3.12"):/# if Version(python_version) < Version("3.9") or Version(python_version) >= Version("3.12"):/g' submodules/TTS/setup.py
    !sed -i 's/    raise RuntimeError("TTS requires python >= 3.9 and < 3.12 " "but your Python version is {}".format(sys.version))/#     raise RuntimeError("TTS requires python >= 3.9 and < 3.12 " "but your Python version is {}".format(sys.version))/g' submodules/TTS/setup.py
    !sed -i 's/python_requires=">=3.9.0, <3.12",/python_requires=">=3.9.0, <3.13",/g' submodules/TTS/setup.py
    print("Patched TTS for Python 3.12 compatibility.")

# 4. 安装项目及子模块依赖
print("Installing project requirements...")
!uv pip install --system -r requirements.txt
!uv pip install --system -r requirements_module.txt

print("Python dependencies installed successfully.")

# 5. 验证关键包
try:
    import yt_dlp
    import loguru
    print("✅ yt-dlp and loguru installed successfully.")
except ImportError as e:
    print(f"❌ Verification failed: {e}. Attempting recovery...")
    !pip install yt-dlp loguru

In [ ]:
# [Step 4] 应用补丁与下载模型 (Apply Patches & Download Models)
# 运行补丁脚本解决 Matplotlib, TTS 和 Videotrans 的兼容性问题
!python patch_colab_fix.py

# 下载核心模型
print("Downloading models...")
!mkdir -p models/ASR/whisper
!wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth \
    -O models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth
!python scripts/huggingface_download.py

Running all Colab patches...
✅ Gradio compatibility handled in webui.py.
Patching submodules/TTS/setup.py for Python 3.12 compatibility...
✅ TTS patched.
Patching Matplotlib backend for headless environment...
✅ Matplotlib backend set to 'Agg'.
⚠️ Could not find videotrans to patch. It might not be installed yet.
Fixing gradio-client version...
ERROR: Ignored the following yanked versions: 0.1.1
ERROR: Could not find a version that satisfies the requirement gradio-client==1.2.7 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6b10, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.1.0, 0.1.2b1, 0.1.2, 0.1.3, 0.1.4, 0.2.0, 0.2.1, 0.2.2, 0.2.3, 0.2.4, 0.2.5, 0.2.6, 0.2.7, 0.2.8, 0.2.9, 0.2.10, 0.3.0, 0.4.0, 0.5.0, 0.5.1, 0.5.2, 0.5.3, 0.6.0b0, 0.6.0b1, 0.6.0b2, 0.6.0, 0.6.1, 0.7.0b0, 0.7.0b2, 0.7.0, 0.7.1, 0.7.2, 0.7.3, 0.8.0, 0.8.1, 0.9.0, 0.10.0, 0.10.1, 0.11.0, 0.12.0, 0.13.0, 0.14.0, 0.15.0, 0.15.1, 0.16.0, 0.16.1, 0.16.2, 0.16.3, 0.16.4, 0.17.0, 1.0.1, 1.0.2, 1.1.0, 1.1.1, 1.2.0, 1.3.0, 1

In [ ]:
# [Step 5] 启动 WebUI (Launch WebUI)
import os
os.environ['MPLBACKEND'] = 'Agg' # Headless Matplotlib

if not os.path.exists('.env'):
    !cp env.example .env

print("Starting WebUI... Please click the Public URL (gradio.live) once it appears.")
!python webui.py

Starting WebUI... Please click the Public URL (gradio.live) once it appears.
Traceback (most recent call last):
  File "/content/Linly-Dubbing/webui.py", line 1, in <module>
    import gradio as gr
ModuleNotFoundError: No module named 'gradio'
